# Test 03: Julia-Only Installation

Integration test for the `sfnb_multilang` toolkit -- Julia only.

**What this tests:**
1. Toolkit installation via pip
2. Julia runtime installation (micromamba + conda-forge)
3. JULIA_PKG_SERVER bypass (the SPCS DNS workaround)
4. Julia package installation (DataFrames, CSV, Arrow, PythonCall)
5. `%%julia` magic registration and execution
6. Python <-> Julia data transfer
7. Snowflake connectivity via Python proxy
8. Optional: ODBC driver
9. Optional: PackageCompiler sysimage

**Prerequisites:**
- EAI with github.com + PyPI access enabled on this notebook
- Note: Julia packages are fetched via Git from github.com (not pkg.julialang.org)

## 1. Install the Toolkit

In [ ]:
# Install from local source (notebooks/ is inside the package dir)
!pip install -q ..

## 2. Install Julia via Toolkit

This replaces `!bash setup_julia_environment.sh`. The toolkit installs
Julia via micromamba, sets `JULIA_PKG_SERVER=""` to bypass the package
server DNS issues, and installs Julia packages via `Pkg.add()`.

In [ ]:
from sfnb_multilang import install

report = install(languages=["julia"])

In [ ]:
print(f"Success: {report.success}")
print(f"Env prefix: {report.env_prefix}")
print(f"Duration: {report.elapsed_seconds:.1f}s")

julia_result = report.plugin_results.get("julia")
if julia_result:
    print(f"Julia version: {julia_result.version}")
    print(f"Julia install OK: {julia_result.success}")
    if julia_result.errors:
        print(f"Errors: {julia_result.errors}")

## 3. Verify JULIA_PKG_SERVER Bypass

Confirm the workaround is in effect. With `JULIA_PKG_SERVER=""`,
Julia clones packages from github.com via Git instead of using
the package server (which has SPCS DNS issues).

In [ ]:
import os

pkg_server = os.environ.get("JULIA_PKG_SERVER", "<not set>")
depot_path = os.environ.get("JULIA_DEPOT_PATH", "<not set>")

print(f"JULIA_PKG_SERVER: '{pkg_server}'")
print(f"JULIA_DEPOT_PATH: {depot_path}")

assert pkg_server == "", f"FAIL: JULIA_PKG_SERVER should be empty, got '{pkg_server}'"
assert depot_path != "<not set>", "FAIL: JULIA_DEPOT_PATH not set"
print("\nPASS: Julia env vars correctly configured")

## 4. Set Up Julia Magic

In [ ]:
from julia_helpers import setup_julia_environment
setup_julia_environment()

## 5. Basic Julia Execution

In [ ]:
%%julia
println("Hello from Julia!")
println("Julia version: ", VERSION)
println("Threads: ", Threads.nthreads())

## 6. Julia Package Availability

In [ ]:
%%julia
using DataFrames
using CSV
using Arrow
using Statistics
using LinearAlgebra

println("PASS: All core Julia packages loaded")

## 7. Julia DataFrames

In [ ]:
%%julia
using DataFrames, Statistics

df = DataFrame(
    name = ["Alice", "Bob", "Carol", "Dave"],
    department = ["Eng", "Sales", "Eng", "Sales"],
    salary = [120000, 95000, 115000, 88000]
)

result = combine(groupby(df, :department),
    :salary => mean => :avg_salary,
    :salary => length => :headcount
)
println(result)

## 8. Python <-> Julia Data Transfer

In [ ]:
import pandas as pd

py_df = pd.DataFrame({
    "x": [1.0, 2.0, 3.0, 4.0, 5.0],
    "y": [2.1, 3.9, 6.2, 7.8, 10.1]
})
print("Python DataFrame:")
print(py_df)

In [ ]:
%%julia -i py_df -o julia_stats
using Statistics, DataFrames

df = DataFrame(py_df)
x = df.x
y = df.y
correlation = cor(x, y)
println("Correlation: ", round(correlation, digits=4))

julia_stats = Dict("correlation" => correlation, "mean_x" => mean(x), "mean_y" => mean(y))

In [ ]:
print("Back in Python:")
print(f"  Correlation: {julia_stats}")
print("\nPASS: Python <-> Julia data transfer works")

## 9. Snowflake Connectivity

Julia accesses Snowflake via the Python Snowpark session proxy.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

sf_df = session.sql("SELECT CURRENT_ACCOUNT() AS acct, CURRENT_WAREHOUSE() AS wh").to_pandas()
print(sf_df)

In [ ]:
%%julia -i sf_df
using DataFrames
df = DataFrame(sf_df)
println("Snowflake data received in Julia:")
println(df)
println("\nPASS: Julia receives Snowflake data via Python proxy")

## 10. (Optional) ODBC Test

Uncomment to test direct Snowflake connectivity via ODBC.jl.

In [ ]:
# To test ODBC, edit julia_only.yaml:
#   snowflake_odbc:
#     enabled: true
# Then re-install:
# from sfnb_multilang import install
# install(config="configs/julia_only.yaml")
#
# %%julia
# using ODBC
# println("ODBC drivers: ", ODBC.drivers())
# println("PASS: ODBC driver available")

## Test Summary

| Test | Expected |
|---|---|
| Toolkit install | pip succeeds |
| Julia install | report.success == True |
| JULIA_PKG_SERVER | Empty string (bypass active) |
| Julia magic | `%%julia` cells execute |
| Packages | DataFrames, CSV, Arrow, Statistics, LinearAlgebra |
| Data transfer | Python <-> Julia round-trip |
| Snowflake | Julia receives data via Python proxy |